In [1]:
# Setup

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import json
import zipfile
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
import torch.nn as nn

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/intentmap-nids/intentmap-nids"
)

DATA_DIR = PROJECT_ROOT / "data"
UNSW_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
CONFIG_DIR = PROJECT_ROOT / "config"
RESULT_DIR = PROJECT_ROOT / "results" / "gns3_zeek"

RESULT_DIR.mkdir(parents=True, exist_ok=True)

X_FILE = DATA_DIR / "combined_practical_41_features.csv"
METADATA_FILE = DATA_DIR / "combined_practical_metadata.csv"
VAL_FILE = UNSW_DIR / "val_normal.npz"

print("41-feature data:", X_FILE.exists())
print("Metadata:", METADATA_FILE.exists())
print("UNSW validation:", VAL_FILE.exists())
print("Result folder:", RESULT_DIR)

Mounted at /content/drive
41-feature data: True
Metadata: True
UNSW validation: True
Result folder: /content/drive/MyDrive/intentmap-nids/intentmap-nids/results/gns3_zeek


In [2]:
# Load processed Zeek data

X_zeek_df = pd.read_csv(X_FILE)
metadata = pd.read_csv(METADATA_FILE)

X_zeek = X_zeek_df.to_numpy(dtype=np.float32)
y_zeek = metadata["label"].to_numpy(dtype=np.int8)

with np.load(VAL_FILE) as data:
    X_val = data["X"].astype(np.float32)

print("Zeek features:", X_zeek.shape)
print("Metadata:", metadata.shape)
print("Labels:", y_zeek.shape)

print("Normal:", int((y_zeek == 0).sum()))
print("Attack:", int((y_zeek == 1).sum()))

print("\nTraffic types:")
print(metadata["traffic_type"].value_counts())

print("\nData sources:")
print(metadata["data_source"].value_counts())

print("\nNaN:", int(np.isnan(X_zeek).sum()))
print("Infinite:", int(np.isinf(X_zeek).sum()))

assert X_zeek.shape[1] == 41
assert len(X_zeek) == len(metadata)
assert len(X_zeek) == len(y_zeek)
assert np.isfinite(X_zeek).all()

print("\nZEEK DATA READY")

Zeek features: (3047, 41)
Metadata: (3047, 9)
Labels: (3047,)
Normal: 16
Attack: 3031

Traffic types:
traffic_type
nmap_scan        2006
dns_burst         600
http_burst        401
ssh_failed         20
normal             16
bulk_transfer       3
icmp_burst          1
Name: count, dtype: int64

Data sources:
data_source
external_laptop    1631
internal_gns3      1416
Name: count, dtype: int64

NaN: 0
Infinite: 0

ZEEK DATA READY


In [3]:
# Model and configuration paths

IF_MODEL_FILE = MODEL_DIR / "isolation_forest_candidate1.joblib"
AE_MODEL_FILE = MODEL_DIR / "autoencoder_candidate1.keras"
LOF_MODEL_FILE = MODEL_DIR / "lof_baseline.joblib"
OCSVM_MODEL_FILE = MODEL_DIR / "ocsvm_model.joblib"
DEEP_MODEL_FILE = MODEL_DIR / "deep_svdd_model.pt"

IF_CONFIG_FILE = CONFIG_DIR / "isolation_forest_candidate1.json"
AE_CONFIG_FILE = CONFIG_DIR / "autoencoder_candidate1.json"
LOF_CONFIG_FILE = CONFIG_DIR / "lof_baseline.json"
OCSVM_CONFIG_FILE = CONFIG_DIR / "ocsvm_model.json"
DEEP_CONFIG_FILE = CONFIG_DIR / "deep_svdd_model.json"
HYBRID_CONFIG_FILE = CONFIG_DIR / "hybrid_if_ae.json"

required_files = [
    IF_MODEL_FILE,
    AE_MODEL_FILE,
    LOF_MODEL_FILE,
    OCSVM_MODEL_FILE,
    DEEP_MODEL_FILE,
    IF_CONFIG_FILE,
    AE_CONFIG_FILE,
    LOF_CONFIG_FILE,
    OCSVM_CONFIG_FILE,
    DEEP_CONFIG_FILE,
    HYBRID_CONFIG_FILE
]

for file in required_files:
    print(file.name, "->", file.exists())

if not all(file.exists() for file in required_files):
    raise FileNotFoundError(
        "One or more trained model/configuration files are missing."
    )

isolation_forest_candidate1.joblib -> True
autoencoder_candidate1.keras -> True
lof_baseline.joblib -> True
ocsvm_model.joblib -> True
deep_svdd_model.pt -> True
isolation_forest_candidate1.json -> True
autoencoder_candidate1.json -> True
lof_baseline.json -> True
ocsvm_model.json -> True
deep_svdd_model.json -> True
hybrid_if_ae.json -> True


In [4]:
# Load model configurations and thresholds

def load_json(path):
    with open(path, "r") as file:
        return json.load(file)

if_config = load_json(IF_CONFIG_FILE)
ae_config = load_json(AE_CONFIG_FILE)
lof_config = load_json(LOF_CONFIG_FILE)
ocsvm_config = load_json(OCSVM_CONFIG_FILE)
deep_config = load_json(DEEP_CONFIG_FILE)
hybrid_config = load_json(HYBRID_CONFIG_FILE)

BUDGETS = [
    "0.5% budget",
    "1% budget",
    "3% budget"
]

def get_thresholds(config):
    return {
        name: float(value)
        for name, value in config["thresholds"].items()
    }

if_thresholds = get_thresholds(if_config)
ae_thresholds = get_thresholds(ae_config)
lof_thresholds = get_thresholds(lof_config)
ocsvm_thresholds = get_thresholds(ocsvm_config)
deep_thresholds = get_thresholds(deep_config)
hybrid_thresholds = get_thresholds(hybrid_config)

print("IF:", if_thresholds)
print("AE:", ae_thresholds)
print("LOF:", lof_thresholds)
print("OCSVM:", ocsvm_thresholds)
print("Deep SVDD:", deep_thresholds)
print("Hybrid:", hybrid_thresholds)

IF: {'0.5% budget': 0.5676495685072951, '1% budget': 0.5460537691550149, '3% budget': 0.4966390963721801}
AE: {'0.5% budget': 0.04577401398215515, '1% budget': 0.036467666841251495, '3% budget': 0.02375445595651247}
LOF: {'0.5% budget': 1.9827458421041482, '1% budget': 1.8009945299909127, '3% budget': 1.476140406733413}
OCSVM: {'0.5% budget': 0.715225019931776, '1% budget': -5.3313550321087755e-05, '3% budget': -1.0914602100972197}
Deep SVDD: {'0.5% budget': 8.080383850028738e-05, '1% budget': 5.488995884661563e-05, '3% budget': 3.0722618248546496e-05}
Hybrid: {'0.5% budget': 0.9909942047336122, '1% budget': 0.9800818155254699, '3% budget': 0.9389378591604168}


In [5]:
# Load trained models

if_model = joblib.load(IF_MODEL_FILE)

ae_model = tf.keras.models.load_model(
    AE_MODEL_FILE,
    compile=False
)

lof_model = joblib.load(LOF_MODEL_FILE)
ocsvm_model = joblib.load(OCSVM_MODEL_FILE)

print("Isolation Forest loaded")
print("Autoencoder loaded")
print("LOF loaded")
print("One-Class SVM loaded")

Isolation Forest loaded
Autoencoder loaded
LOF loaded
One-Class SVM loaded


In [6]:
# Score IF, AE, LOF and OCSVM

if_zeek_scores = -if_model.score_samples(X_zeek)
if_val_scores = -if_model.score_samples(X_val)

ae_zeek_reconstructed = ae_model.predict(
    X_zeek,
    batch_size=1024,
    verbose=0
)

ae_zeek_scores = np.mean(
    np.square(
        X_zeek - ae_zeek_reconstructed
    ),
    axis=1
)

ae_val_reconstructed = ae_model.predict(
    X_val,
    batch_size=1024,
    verbose=0
)

ae_val_scores = np.mean(
    np.square(
        X_val - ae_val_reconstructed
    ),
    axis=1
)

lof_zeek_scores = -lof_model.score_samples(
    X_zeek
)

ocsvm_zeek_scores = -ocsvm_model.decision_function(
    X_zeek
).reshape(-1)

print("IF:", if_zeek_scores.shape)
print("AE:", ae_zeek_scores.shape)
print("LOF:", lof_zeek_scores.shape)
print("OCSVM:", ocsvm_zeek_scores.shape)

IF: (3047,)
AE: (3047,)
LOF: (3047,)
OCSVM: (3047,)


In [7]:
# Load and score Deep SVDD

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

class DeepSVDDNet(nn.Module):
    def __init__(self, input_dim, representation_dim=16):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64, bias=False),
            nn.ReLU(),
            nn.Linear(64, 32, bias=False),
            nn.ReLU(),
            nn.Linear(32, representation_dim, bias=False)
        )

    def forward(self, x):
        return self.network(x)

checkpoint = torch.load(
    DEEP_MODEL_FILE,
    map_location=DEVICE,
    weights_only=False
)

deep_model = DeepSVDDNet(
    input_dim=int(checkpoint["input_dim"]),
    representation_dim=int(checkpoint["representation_dim"])
).to(DEVICE)

deep_model.load_state_dict(
    checkpoint["model_state_dict"]
)

deep_model.eval()

deep_center = torch.tensor(
    checkpoint["center"],
    dtype=torch.float32,
    device=DEVICE
)

with torch.no_grad():

    X_tensor = torch.tensor(
        X_zeek,
        dtype=torch.float32,
        device=DEVICE
    )

    output = deep_model(X_tensor)

    deep_zeek_scores = torch.sum(
        (output - deep_center) ** 2,
        dim=1
    ).cpu().numpy()

print("Deep SVDD:", deep_zeek_scores.shape)
print("Device:", DEVICE)

Deep SVDD: (3047,)
Device: cpu


In [8]:
# Hybrid IF + AE scores

def percentile_normalize(reference_scores, scores):
    reference_sorted = np.sort(
        np.asarray(reference_scores)
    )

    ranks = np.searchsorted(
        reference_sorted,
        scores,
        side="right"
    )

    return (
        ranks /
        (len(reference_sorted) + 1)
    )

if_val_norm = percentile_normalize(
    if_val_scores,
    if_val_scores
)

if_zeek_norm = percentile_normalize(
    if_val_scores,
    if_zeek_scores
)

ae_val_norm = percentile_normalize(
    ae_val_scores,
    ae_val_scores
)

ae_zeek_norm = percentile_normalize(
    ae_val_scores,
    ae_zeek_scores
)

IF_WEIGHT = float(
    hybrid_config["if_weight"]
)

AE_WEIGHT = float(
    hybrid_config["ae_weight"]
)

hybrid_zeek_scores = (
    IF_WEIGHT * if_zeek_norm
    +
    AE_WEIGHT * ae_zeek_norm
)

print("IF weight:", IF_WEIGHT)
print("AE weight:", AE_WEIGHT)

print(
    "Hybrid score range:",
    hybrid_zeek_scores.min(),
    hybrid_zeek_scores.max()
)

IF weight: 0.5
AE weight: 0.5
Hybrid score range: 0.9069348397779293 0.9999026005649168


In [9]:
# Evaluation function

def evaluate_model(
    model_name,
    scores,
    thresholds
):
    rows = []

    roc_auc = roc_auc_score(
        y_zeek,
        scores
    )

    pr_auc = average_precision_score(
        y_zeek,
        scores
    )

    for budget in BUDGETS:

        threshold = float(
            thresholds[budget]
        )

        predictions = (
            scores >= threshold
        ).astype(np.int8)

        tn, fp, fn, tp = confusion_matrix(
            y_zeek,
            predictions,
            labels=[0, 1]
        ).ravel()

        precision = precision_score(
            y_zeek,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_zeek,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_zeek,
            predictions,
            zero_division=0
        )

        fpr = (
            fp / (fp + tn)
            if (fp + tn) > 0
            else 0
        )

        rows.append({
            "Model": model_name,
            "Budget": budget,
            "Threshold": threshold,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "FPR": fpr,
            "False alerts per 1000": fpr * 1000,
            "ROC-AUC": roc_auc,
            "PR-AUC": pr_auc,
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp)
        })

    return pd.DataFrame(rows)

In [10]:
# Evaluate all six models

if_results = evaluate_model(
    "Isolation Forest",
    if_zeek_scores,
    if_thresholds
)

ae_results = evaluate_model(
    "Dense Autoencoder",
    ae_zeek_scores,
    ae_thresholds
)

hybrid_results = evaluate_model(
    "Hybrid IF+AE",
    hybrid_zeek_scores,
    hybrid_thresholds
)

lof_results = evaluate_model(
    "LOF",
    lof_zeek_scores,
    lof_thresholds
)

ocsvm_results = evaluate_model(
    "One-Class SVM",
    ocsvm_zeek_scores,
    ocsvm_thresholds
)

deep_results = evaluate_model(
    "Deep SVDD",
    deep_zeek_scores,
    deep_thresholds
)

all_results = pd.concat(
    [
        if_results,
        ae_results,
        hybrid_results,
        lof_results,
        ocsvm_results,
        deep_results
    ],
    ignore_index=True
)

display(all_results)

,Model,Budget,Threshold,Precision,Recall,F1,FPR,False alerts per 1000,ROC-AUC,PR-AUC,TN,FP,FN,TP
0,Isolation Forest,0.5% budget,0.567650,0.997684,0.995051,0.996366,0.4375,437.5,0.984184,0.999915,9,7,15,3016
1,Isolation Forest,1% budget,0.546054,0.997357,0.996041,0.996699,0.5000,500.0,0.984184,0.999915,8,8,12,3019
2,Isolation Forest,3% budget,0.496639,0.995725,0.999010,0.997365,0.8125,812.5,0.984184,0.999915,3,13,3,3028
3,Dense Autoencoder,0.5% budget,0.045774,0.996376,0.997691,0.997033,0.6875,687.5,0.927417,0.998871,5,11,7,3024
4,Dense Autoencoder,1% budget,0.036468,0.996377,0.998020,0.997198,0.6875,687.5,0.927417,0.998871,5,11,6,3025
5,Dense Autoencoder,3% budget,0.023754,0.996384,1.000000,0.998189,0.6875,687.5,0.927417,0.998871,5,11,0,3031
6,Hybrid IF+AE,0.5% budget,0.990994,0.996701,0.996701,0.996701,0.6250,625.0,0.985081,0.999904,6,10,10,3021
7,Hybrid IF+AE,1% budget,0.980082,0.996379,0.998680,0.997528,0.6875,687.5,0.985081,0.999904,5,11,4,3027
8,Hybrid IF+AE,3% budget,0.938938,0.995402,1.000000,0.997696,0.8750,875.0,0.985081,0.999904,2,14,0,3031
9,LOF,0.5% budget,1.982746,0.997363,0.998350,0.997857,0.5000,500.0,0.814088,0.995623,8,8,5,3026


In [11]:
# Final 1% comparison

comparison_1pct = (
    all_results[
        all_results["Budget"] == "1% budget"
    ]
    .copy()
    .sort_values(
        "F1",
        ascending=False
    )
    .reset_index(drop=True)
)

comparison_1pct["FPR (%)"] = (
    comparison_1pct["FPR"] * 100
)

comparison_1pct = comparison_1pct[
    [
        "Model",
        "Precision",
        "Recall",
        "F1",
        "FPR (%)",
        "False alerts per 1000",
        "ROC-AUC",
        "PR-AUC",
        "TN",
        "FP",
        "FN",
        "TP"
    ]
]

display(comparison_1pct)

,Model,Precision,Recall,F1,FPR (%),False alerts per 1000,ROC-AUC,PR-AUC,TN,FP,FN,TP
0,LOF,0.996707,0.998680,0.997693,62.50,625.0,0.814088,0.995623,6,10,4,3027
1,Hybrid IF+AE,0.996379,0.998680,0.997528,68.75,687.5,0.985081,0.999904,5,11,4,3027
2,Dense Autoencoder,0.996377,0.998020,0.997198,68.75,687.5,0.927417,0.998871,5,11,6,3025
3,Isolation Forest,0.997357,0.996041,0.996699,50.00,500.0,0.984184,0.999915,8,8,12,3019
4,Deep SVDD,0.995063,0.997361,0.996210,93.75,937.5,0.816026,0.995388,1,15,8,3023
5,One-Class SVM,0.997678,0.992412,0.995038,43.75,437.5,0.917684,0.998035,9,7,23,3008


In [12]:
# Create 1% predictions

if_pred = (
    if_zeek_scores >= if_thresholds["1% budget"]
).astype(np.int8)

ae_pred = (
    ae_zeek_scores >= ae_thresholds["1% budget"]
).astype(np.int8)

hybrid_pred = (
    hybrid_zeek_scores >= hybrid_thresholds["1% budget"]
).astype(np.int8)

lof_pred = (
    lof_zeek_scores >= lof_thresholds["1% budget"]
).astype(np.int8)

ocsvm_pred = (
    ocsvm_zeek_scores >= ocsvm_thresholds["1% budget"]
).astype(np.int8)

deep_pred = (
    deep_zeek_scores >= deep_thresholds["1% budget"]
).astype(np.int8)

print("Predictions created")

Predictions created


In [13]:
# Per-scenario detection results

prediction_map = {
    "Isolation Forest": if_pred,
    "Dense Autoencoder": ae_pred,
    "Hybrid IF+AE": hybrid_pred,
    "LOF": lof_pred,
    "One-Class SVM": ocsvm_pred,
    "Deep SVDD": deep_pred
}

scenario_rows = []

for traffic_type in metadata["traffic_type"].unique():

    mask = (
        metadata["traffic_type"] == traffic_type
    ).to_numpy()

    actual_label = int(
        metadata.loc[
            metadata["traffic_type"] == traffic_type,
            "label"
        ].iloc[0]
    )

    for model_name, predictions in prediction_map.items():

        detection_rate = predictions[mask].mean()

        scenario_rows.append({
            "Traffic Type": traffic_type,
            "Actual Label": actual_label,
            "Records": int(mask.sum()),
            "Model": model_name,
            "Detection Rate": detection_rate
        })

scenario_results = pd.DataFrame(
    scenario_rows
)

display(scenario_results)

,Traffic Type,Actual Label,Records,Model,Detection Rate
0,bulk_transfer,1,3,Isolation Forest,1.000000
1,bulk_transfer,1,3,Dense Autoencoder,1.000000
2,bulk_transfer,1,3,Hybrid IF+AE,1.000000
3,bulk_transfer,1,3,LOF,0.333333
4,bulk_transfer,1,3,One-Class SVM,1.000000
5,bulk_transfer,1,3,Deep SVDD,1.000000
6,dns_burst,1,600,Isolation Forest,0.991667
7,dns_burst,1,600,Dense Autoencoder,1.000000
8,dns_burst,1,600,Hybrid IF+AE,1.000000
9,dns_burst,1,600,LOF,1.000000


In [14]:
# Internal vs external results

source_rows = []

for source in metadata["data_source"].unique():

    source_mask = (
        metadata["data_source"] == source
    ).to_numpy()

    for model_name, predictions in prediction_map.items():

        attack_mask = (
            source_mask &
            (y_zeek == 1)
        )

        normal_mask = (
            source_mask &
            (y_zeek == 0)
        )

        attack_detection = (
            predictions[attack_mask].mean()
            if attack_mask.sum() > 0
            else np.nan
        )

        normal_false_positive = (
            predictions[normal_mask].mean()
            if normal_mask.sum() > 0
            else np.nan
        )

        source_rows.append({
            "Data Source": source,
            "Model": model_name,
            "Attack Records": int(attack_mask.sum()),
            "Normal Records": int(normal_mask.sum()),
            "Attack Detection Rate": attack_detection,
            "Normal False Positive Rate": normal_false_positive
        })

source_results = pd.DataFrame(
    source_rows
)

display(source_results)

,Data Source,Model,Attack Records,Normal Records,Attack Detection Rate,Normal False Positive Rate
0,internal_gns3,Isolation Forest,1404,12,0.991453,0.333333
1,internal_gns3,Dense Autoencoder,1404,12,0.995726,0.583333
2,internal_gns3,Hybrid IF+AE,1404,12,0.997151,0.583333
3,internal_gns3,LOF,1404,12,0.997151,0.500000
4,internal_gns3,One-Class SVM,1404,12,0.983618,0.250000
5,internal_gns3,Deep SVDD,1404,12,0.994302,0.916667
6,external_laptop,Isolation Forest,1627,4,1.000000,1.000000
7,external_laptop,Dense Autoencoder,1627,4,1.000000,1.000000
8,external_laptop,Hybrid IF+AE,1627,4,1.000000,1.000000
9,external_laptop,LOF,1627,4,1.000000,1.000000


In [15]:
# Save record-level predictions

record_results = metadata.copy()

record_results["if_score"] = if_zeek_scores
record_results["if_prediction"] = if_pred

record_results["ae_score"] = ae_zeek_scores
record_results["ae_prediction"] = ae_pred

record_results["hybrid_score"] = hybrid_zeek_scores
record_results["hybrid_prediction"] = hybrid_pred

record_results["lof_score"] = lof_zeek_scores
record_results["lof_prediction"] = lof_pred

record_results["ocsvm_score"] = ocsvm_zeek_scores
record_results["ocsvm_prediction"] = ocsvm_pred

record_results["deep_svdd_score"] = deep_zeek_scores
record_results["deep_svdd_prediction"] = deep_pred

RECORD_FILE = (
    RESULT_DIR /
    "GNS3_Zeek_Record_Level_Predictions.csv"
)

record_results.to_csv(
    RECORD_FILE,
    index=False
)

print(RECORD_FILE)

/content/drive/MyDrive/intentmap-nids/intentmap-nids/results/gns3_zeek/GNS3_Zeek_Record_Level_Predictions.csv


In [16]:
# Save report values to Excel

EXCEL_FILE = (
    RESULT_DIR /
    "GNS3_Zeek_Model_Evaluation.xlsx"
)

with pd.ExcelWriter(EXCEL_FILE) as writer:

    comparison_1pct.to_excel(
        writer,
        sheet_name="Model Comparison",
        index=False
    )

    all_results.to_excel(
        writer,
        sheet_name="All Budgets",
        index=False
    )

    scenario_results.to_excel(
        writer,
        sheet_name="Per Scenario",
        index=False
    )

    source_results.to_excel(
        writer,
        sheet_name="Internal External",
        index=False
    )

print("Excel saved:")
print(EXCEL_FILE)

Excel saved:
/content/drive/MyDrive/intentmap-nids/intentmap-nids/results/gns3_zeek/GNS3_Zeek_Model_Evaluation.xlsx


In [19]:
# Create ZIP and download

from google.colab import files

ZIP_FILE = (
    RESULT_DIR /
    "GNS3_Zeek_Testing_Results.zip"
)

with zipfile.ZipFile(
    ZIP_FILE,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for file in [
        EXCEL_FILE,
        RECORD_FILE,
        BAR_CHART_FILE,
        SCENARIO_CHART_FILE
    ]:
        zip_file.write(
            file,
            arcname=file.name
        )

print("ZIP created:")
print(ZIP_FILE)

files.download(
    str(ZIP_FILE)
)

ZIP created:
/content/drive/MyDrive/intentmap-nids/intentmap-nids/results/gns3_zeek/GNS3_Zeek_Testing_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>